# 12 · Dative SMILES and CXSMILES

`dative_smiles(mol)` writes the canonical constitution; pass it to `metal` with a geometry to enumerate
arrangements. `cxsmiles(source)` also records one arrangement, so `embed` reads it directly. Use
`parse_smiles` only when an RDKit `Mol` is needed without embedding.

In [ ]:
from rdkit import Chem, RDLogger

import rxembed as rx

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

cisplatin = Chem.AddHs(Chem.MolFromSmiles("N->[Pt+2](<-N)(<-[Cl-])<-[Cl-]"))
print("dative SMILES:", rx.dative_smiles(cisplatin))

## Constitution and arrangement

`dative_smiles` contains connectivity, charges and ligand stereocentres. It deliberately omits the metal
arrangement, so cis and trans have the same constitution string.

`cxsmiles` adds an RDKit CXSMILES `atomProp` block. A value such as `1.atomNote.s2` assigns atom position 1
in the preceding SMILES to canonical donor slot 2; `4.atomNote.SPL` labels atom position 4 as square planar.
The slots and any Λ/Δ label distinguish arrangements supported by rxembed. RDKit parses and preserves the
properties but does not interpret the slot semantics; rxembed does that when embedding. The writer first
canonicalizes the graph, so the numbers are canonical-SMILES positions, not input atom indices.
Planar-chiral haptic winding enumerates as `ηn+`/`ηn-`, is stored on the face slot, and is selected immediately after distance geometry. Use `stereo="free"` to leave it unspecified.

In [ ]:
isos = rx.enumerate_isomers(cisplatin, "square_planar")
for iso in isos:
    print(f"{iso.label:6s} |atomProp:{rx.cxsmiles(iso).split('|atomProp:')[1]}")

cores = {rx.cxsmiles(i).split(" |", 1)[0] for i in isos}
print(f"\n{len(isos)} arrangements, {len(cores)} constitution:", cores == {rx.dative_smiles(cisplatin)})

## Lewis-form normalization

rxembed accepts metal SMILES in either neutral/covalent or ionic dative form. The neutral strings below
are deliberate demonstrations of that input support; examples elsewhere use ionic dative SMILES.
`dative_smiles` derives the common form from donor valence. Bond direction and formal charge are separate:
neutral ammine is dative without becoming charged, while chloride becomes `[Cl-]->[M+]`. Each pair below
therefore produces one string.

In [ ]:
one_species = {
    "chloride": ("N->[Pt](<-N)(Cl)Cl", "N->[Pt+2](<-N)(<-[Cl-])<-[Cl-]"),
    "phosphine + anion": ("CP(C)(C)->[Rh]Cl", "CP(C)(C)->[Rh+]<-[Cl-]"),
    "terminal oxo": ("O=[V](Cl)(Cl)Cl", "[O-2]->[V+5](<-[Cl-])(<-[Cl-])<-[Cl-]"),
}
for kind, pair in one_species.items():
    written = {rx.dative_smiles(Chem.AddHs(Chem.MolFromSmiles(s))) for s in pair}
    print(f"{kind:18s} {len(written)} string   {written.pop()}")

Ambiguous neutral donors remain distinct. Both Lewis forms below are valid, so the writer does not guess.

In [ ]:
partly_filled = {
    "amide": ("CN(C)[Pd](Cl)Cl", "C[N-](C)->[Pd+3](<-[Cl-])<-[Cl-]"),
    "alkyl": ("C[Pd](Cl)(Cl)C", "[CH3-]->[Pd+4](<-[Cl-])(<-[Cl-])<-[CH3-]"),
}
for kind, pair in partly_filled.items():
    written = {rx.dative_smiles(Chem.AddHs(Chem.MolFromSmiles(s))) for s in pair}
    print(f"{kind:6s} {len(written)} strings")
    for s in sorted(written):
        print("      ", s)

## Read and embed

Use `rx.embed(text)` for the normal round trip. `rx.parse_smiles(text)` only returns the `Mol`. Routine
hydrogens are implicit; hydrides and other hydrogen donors stay explicit when they carry a slot.

In [ ]:
text = rx.cxsmiles(isos[0])  # the cis isomer
embedded = rx.embed(text, n=2, seed=1)
print("isomer read back:", embedded.iso)
print("fixed point:", rx.cxsmiles(embedded.iso) == text)

mol = rx.parse_smiles(text)
print(f"parse_smiles returned an RDKit Mol with {mol.GetNumAtoms()} atoms")

## Preserve the stated arrangement

Distance and angle constraints do not distinguish mirror images. After minimization, rxembed measures the
coordination sphere and corrects the hand. Any unresolved conformer id is recorded in `wrong_hand`.

In [ ]:
ma2b2c2 = Chem.AddHs(Chem.MolFromSmiles("[F-]->[Pt+4](<-[F-])(<-[Cl-])(<-[Cl-])(<-[Br-])<-[Br-]"))
hands = [i for i in rx.enumerate_isomers(ma2b2c2, "octahedral") if i.chirality]

for iso in hands:
    stated = rx.cxsmiles(iso)
    confs = rx.embed(stated, n=4, seed=1).minimize()
    print(f"{iso.chirality:7s} {confs.n} conformers; wrong hand: {confs.wrong_hand or 'none'}")

## Read coordinates

`read_xyz(path)` reads coordinates and perceives an RDKit molecular graph. The Ru pincer below closes the
full `.xyz -> Mol -> CXSMILES -> embed` path. Two donors share slot `s1` because its η² face is one vertex.

In [ ]:
ru = rx.read_xyz("structures/ru-co.xyz")
ru_text = rx.cxsmiles(ru)
core = ru_text.split(" |", 1)[0]
embedded_ru = rx.embed(ru_text, n=1)
rep = embedded_ru.check()[embedded_ru.ids[0]]
print("geometry.check:", "CLEAN" if rep.ok() else f"FAIL {rep.violations[:4]}")
print(f"\nru-co.xyz: {ru.GetNumAtoms()} atoms")
print("dative SMILES:", core)
print("arrangement:", embedded_ru.iso)
print("  fixed point:", rx.cxsmiles(embedded_ru.iso) == ru_text)